# 02 — Base Model Baseline

Measure the base model before adapter training. Baseline and fine-tuned models must use the same prompts, decoding parameters, evaluation cases, and runtime conditions.

In [ ]:
import os, json, time, statistics
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0

## Evaluation prompts

Keep prompts in a versioned file and ensure they are permitted, de-identified, and research-oriented. Avoid presenting model outputs as clinical advice.

In [ ]:
evaluation_prompts = [
    "Explain what hypertension means in general educational terms.",
    "Describe common warning signs that require urgent professional evaluation.",
    "Explain the difference between a symptom and a diagnosis.",
]

## Load model

Model loading can require substantial GPU RAM. Use a smaller checkpoint or cloud GPU if memory is insufficient.

In [ ]:
def load_base_model(model_id=MODEL_ID, dtype=None):
    dtype = dtype or (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
                       else torch.float16 if torch.cuda.is_available()
                       else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

# model, tokenizer = load_base_model()

## Prompt formatting

In [ ]:
def format_prompt(question, tokenizer):
    messages = [
        {"role": "system", "content": "You are an educational healthcare assistant. Be cautious, state uncertainty, and do not replace professional care."},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# print(format_prompt(evaluation_prompts[0], tokenizer))

## Generation helper

In [ ]:
@torch.inference_mode()
def generate_answer(model, tokenizer, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = format_prompt(question, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.perf_counter()
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    elapsed = time.perf_counter() - start
    new_tokens = generated[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), elapsed, len(new_tokens)

# answer, latency, token_count = generate_answer(model, tokenizer, evaluation_prompts[0])
# print(answer, latency, token_count)

## Run baseline evaluation

In [ ]:
def run_generation_suite(model, tokenizer, prompts):
    rows = []
    for idx, question in enumerate(prompts):
        answer, elapsed, n_tokens = generate_answer(model, tokenizer, question)
        rows.append({
            "case_id": idx,
            "question": question,
            "answer": answer,
            "latency_seconds": elapsed,
            "generated_tokens": n_tokens,
            "tokens_per_second": n_tokens / elapsed if elapsed else None,
        })
    return pd.DataFrame(rows)

# baseline_results = run_generation_suite(model, tokenizer, evaluation_prompts)
# baseline_results.to_csv("../results/base_model_predictions.csv", index=False)

## Runtime baseline

In [ ]:
def runtime_snapshot():
    snapshot = {"cpu_percent": psutil.cpu_percent(interval=0.5),
                "ram_used_gb": psutil.virtual_memory().used / 1024**3}
    if torch.cuda.is_available():
        snapshot.update({
            "gpu_allocated_gb": torch.cuda.memory_allocated() / 1024**3,
            "gpu_reserved_gb": torch.cuda.memory_reserved() / 1024**3,
        })
    return snapshot

# runtime_snapshot()

## Notes and limitations

Baseline responses are model-generated and must be reviewed against a rubric. A similarity score alone is not a clinical correctness measure.

## Verified local environment check

Actual environment inspection only; this is not a model benchmark.


In [1]:
import platform, importlib.util, torch
packages = ['torch', 'transformers', 'datasets', 'peft', 'trl', 'pandas', 'psutil']
print('Python version:', platform.python_version())
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available in this execution environment')
print('Package availability:')
for package in packages:
    print(f'  {package}: {"installed" if importlib.util.find_spec(package) else "not installed"}')


Python version: 3.13.5
PyTorch version: 2.10.0+cpu
CUDA available: False
GPU: Not available in this execution environment
Package availability:
  torch: installed
  transformers: not installed
  datasets: not installed
  peft: not installed
  trl: not installed
  pandas: installed
  psutil: installed

## Prompt-set validation

The next output validates the prompt strings defined in this notebook. It is not an LLM evaluation.


In [2]:
assert len(evaluation_prompts) > 0
assert all(isinstance(prompt, str) and prompt.strip() for prompt in evaluation_prompts)
print('Number of evaluation prompts:', len(evaluation_prompts))
print('All prompts are non-empty:', all(isinstance(x, str) and x.strip() for x in evaluation_prompts))
print('Prompt IDs:', ', '.join(map(str, range(len(evaluation_prompts)))))
print('Prompt character counts:', ', '.join(str(len(x)) for x in evaluation_prompts))


Number of evaluation prompts: 3
All prompts are non-empty: True
Prompt IDs: 0, 1, 2
Prompt character counts: 61, 74, 57

## Baseline execution status

**Not run:** the model loading and generation cells require `transformers` and model weights. This execution environment has no CUDA GPU and does not have `transformers` installed. Run those cells in a configured runtime to capture actual generated answers and latency measurements.
